# 🌱 Kaggle S6E4 — Predicting Irrigation Need
**Metric:** Balanced Accuracy | **Classes:** Low / Medium / High | **Deadline:** 30. April 2026

Pipeline: EDA → Feature Engineering → LightGBM + CatBoost (5-Fold CV) → Weighted Ensemble → Submission

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from scipy.optimize import minimize
from itertools import combinations

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
# ── Run-Profile ──
# Schnell:     SEEDS=[42],         N_SPLITS=5, ITERS=1000  (~2h Kaggle)
# Mittel:      SEEDS=[42, 2024],   N_SPLITS=5, ITERS=1200  (~4h Kaggle)
# Final:       SEEDS=[42, 2024, 7],N_SPLITS=5, ITERS=1500  (~7h Kaggle)
SEEDS    = [42]
N_SPLITS = 5
N_ITERS  = 1200
TARGET   = 'Irrigation_Need'

USE_PSEUDO_LABELING   = False
PSEUDO_CONF_THRESHOLD = 0.97
USE_STACKING          = True   # LogReg meta-learner on top of LGB+CB+XGB

print('Libraries loaded ✅')
print(f'Profil: {len(SEEDS)} seeds × {N_SPLITS} folds × {N_ITERS} iters')
print(f'Stacking: {USE_STACKING} | Pseudo-labels: {USE_PSEUDO_LABELING}')

## 1. Load Data

In [ ]:
import os, glob

# ── Auto-detect Kaggle vs. local environment ──
KAGGLE_PATH = '/kaggle/input/competitions/playground-series-s6e4'
if os.path.exists(KAGGLE_PATH):
    DATA_DIR = KAGGLE_PATH
else:
    DATA_DIR = '.'
print(f'DATA_DIR: {DATA_DIR}')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

# ── Find external miadul dataset anywhere under /kaggle/input ──
# Kaggle mounts additional datasets at /kaggle/input/<slug>/ — the exact slug
# depends on how the user added it. We search recursively for the CSV.
external_loaded = False
search_paths = [
    '/kaggle/input/**/irrigation_prediction.csv',
    '/kaggle/input/**/irrigation*prediction*.csv',
    '/kaggle/input/**/*irrigation_water*.csv',
    './irrigation_prediction.csv',
    './external/irrigation_prediction.csv',
]
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        external_path = matches[0]
        external_loaded = True
        print(f'✅ External dataset found: {external_path}')
        break

if external_loaded:
    original = pd.read_csv(external_path)
    # Flag samples by origin
    train['is_generated']    = 1
    test['is_generated']     = 1
    original['is_generated'] = 0
    # Combine: only columns present in both
    common_cols = [c for c in train.columns if c in original.columns]
    print(f'   rows: {original.shape[0]}, matching cols: {len(common_cols)}')
    train = pd.concat([train, original[common_cols]], ignore_index=True)
    initial_len = len(train)
    train = train.drop_duplicates().reset_index(drop=True)
    dropped = initial_len - len(train)
    if dropped:
        print(f'   dropped {dropped} duplicate rows')
else:
    print('⚠️  External miadul dataset NOT found — running on synthetic data only.')
    print('   To add it: Kaggle Notebook → Add Input → search "miadul irrigation"')
    # Do NOT add the is_generated column — it would be a constant feature.

print(f'\nTrain: {train.shape}')
print(f'Test:  {test.shape}')
print(f'Sub:   {sub.shape}')
if 'is_generated' in train.columns:
    print(f"is_generated dist: {train['is_generated'].value_counts().to_dict()}")
train.head()

## 2. EDA

In [ ]:
print('=== Data Types ===')
print(train.dtypes)
print('\n=== Missing Values ===')
print(train.isnull().sum())
print('\n=== Target Distribution ===')
print(train[TARGET].value_counts())
print(train[TARGET].value_counts(normalize=True).round(3))

In [ ]:
print('=== Numeric Summary ===')
train.describe().T

In [ ]:
# Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = train[TARGET].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2ecc71','#f39c12','#e74c3c'])
axes[0].set_title('Target Distribution (absolute)', fontsize=13)
axes[0].set_xlabel('Irrigation Need')
axes[0].set_ylabel('Count')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 100, f'{val:,}', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71','#f39c12','#e74c3c'], startangle=90)
axes[1].set_title('Target Distribution (%)', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric features)
num_cols = train.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != 'id']

if len(num_cols) > 1:
    plt.figure(figsize=(14, 10))
    corr = train[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='RdYlGn', center=0,
                annot=len(num_cols) <= 15, fmt='.2f', linewidths=0.5)
    plt.title('Feature Correlation Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution of numeric features by target
num_features = [c for c in num_cols if c not in ['id']]

n_cols = 3
n_rows = (len(num_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()

colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

for i, col in enumerate(num_features):
    ax = axes[i]
    for label in train[TARGET].unique():
        subset = train[train[TARGET] == label][col].dropna()
        subset.hist(ax=ax, bins=40, alpha=0.6, label=label,
                    color=colors.get(label, 'gray'), density=True)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Irrigation Need Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features overview
cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET]
print(f'Categorical columns: {cat_cols}')
for col in cat_cols:
    print(f'\n{col}: {train[col].nunique()} unique values')
    print(train[col].value_counts().head(10))

## 3. Feature Engineering

In [ ]:
# ============================================================
# Advanced Feature Engineering
# Inspiriert von:
# - ravi20076 TunedBlend (LB 0.98018) — Ensemble-Strategie
# - yunsuxiaozi XGB OrderedTE (CV 0.97981) — extended digit features + freq encoding
# - utaazu pairwise CB (LB 0.979)
# ============================================================

BASE_NUM_COLS = [
    'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
    'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
    'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm'
]

THRESHOLDS = {
    'Soil_Moisture': 24.995, 'Wind_Speed_kmh': 9.995,
    'Temperature_C': 30.205, 'Rainfall_mm': 299.480,
    'Previous_Irrigation_mm': 17.974, 'Humidity': 90.990,
    'Electrical_Conductivity': 0.755, 'Field_Area_hectare': 7.545,
    'Soil_pH': 6.455, 'Organic_Carbon': 0.415, 'Sunlight_Hours': 10.395,
}


def add_base_features(df):
    df = df.copy()
    for col, th in THRESHOLDS.items():
        if col in df.columns:
            df[f'is_{col}'] = (df[col] >= th).astype('int8')
    if 'Soil_Moisture' in df.columns:
        df['log_Soil_Moisture']  = np.log1p(df['Soil_Moisture'].clip(lower=0))
        df['inv_Soil_Moisture']  = 1.0 / (df['Soil_Moisture'].abs() + 1e-6)
        df['Soil_Moisture_sq']   = df['Soil_Moisture'] ** 2
    if 'Rainfall_mm' in df.columns:
        df['log_Rainfall_mm']    = np.log1p(df['Rainfall_mm'].clip(lower=0))
        df['inv_Rainfall_mm']    = 1.0 / (df['Rainfall_mm'].abs() + 1e-6)
        df['Rainfall_mm_sq']     = df['Rainfall_mm'] ** 2
    if 'Temperature_C' in df.columns:
        df['Temperature_C_sq']   = df['Temperature_C'] ** 2
    if 'Wind_Speed_kmh' in df.columns:
        df['Wind_Speed_kmh_sq']  = df['Wind_Speed_kmh'] ** 2
    if {'Wind_Speed_kmh', 'Temperature_C'}.issubset(df.columns):
        df['evaporation'] = df['Wind_Speed_kmh'] * df['Temperature_C']
    if 'evaporation' in df.columns:
        if 'Rainfall_mm' in df.columns:
            df['water_balance']   = df['Rainfall_mm'] - df['evaporation']
        if 'Soil_Moisture' in df.columns:
            df['moisture_stress'] = df['Soil_Moisture'] * df['evaporation']
    if {'Rainfall_mm', 'Wind_Speed_kmh'}.issubset(df.columns):
        df['rain_efficiency'] = df['Rainfall_mm'] / (df['Wind_Speed_kmh'] + 1)
        df['rain_wind']       = df['Rainfall_mm'] * df['Wind_Speed_kmh']
    if {'Rainfall_mm', 'Temperature_C'}.issubset(df.columns):
        df['rain_temp']       = df['Rainfall_mm'] * df['Temperature_C']
    if {'Temperature_C', 'Humidity'}.issubset(df.columns):
        df['dryness_index']   = df['Temperature_C'] * (100 - df['Humidity']) / 100
        df['heat_humidity']   = df['Temperature_C'] + 0.5 * df['Humidity']
    if {'Sunlight_Hours', 'Rainfall_mm'}.issubset(df.columns):
        df['sun_rain_ratio']  = df['Sunlight_Hours'] / (df['Rainfall_mm'].abs() + 1e-6)
    if {'Previous_Irrigation_mm', 'Rainfall_mm'}.issubset(df.columns):
        df['total_water']     = df['Previous_Irrigation_mm'] + df['Rainfall_mm']
    if {'Previous_Irrigation_mm', 'Soil_Moisture'}.issubset(df.columns):
        df['prev_moisture_ratio'] = df['Previous_Irrigation_mm'] / (df['Soil_Moisture'] + 1e-6)
    if {'Rainfall_mm', 'Soil_Moisture'}.issubset(df.columns):
        df['rain_moisture_ratio'] = df['Rainfall_mm'] / (df['Soil_Moisture'] + 1e-6)
    if {'Organic_Carbon', 'Electrical_Conductivity'}.issubset(df.columns):
        df['soil_quality'] = df['Organic_Carbon'] * df['Electrical_Conductivity']
    if {'Soil_pH', 'Organic_Carbon'}.issubset(df.columns):
        df['ph_carbon']    = df['Soil_pH'] * df['Organic_Carbon']
    if 'Humidity' in df.columns:
        df['Humidity_rounded'] = ((df['Humidity'] / 10).round() * 10).astype('int8')
    return df


def fit_tree_bins(df, y, cols, max_leaf=10, min_samples=0.01):
    edges = {}
    for col in cols:
        if col not in df.columns: continue
        Xc = df[[col]].values
        if len(np.unique(Xc)) <= 1: continue
        tree = DecisionTreeClassifier(
            max_leaf_nodes=max_leaf, min_samples_leaf=min_samples,
            random_state=42, class_weight='balanced'
        )
        tree.fit(Xc, y)
        th = np.unique(tree.tree_.threshold[tree.tree_.threshold != -2])
        edges[col] = [-np.inf] + sorted(th.tolist()) + [np.inf]
    return edges


def apply_tree_bins(df, edges):
    df = df.copy()
    for col, bins in edges.items():
        if col in df.columns:
            df[f'{col}_tbin'] = pd.cut(df[col], bins=bins, labels=False).fillna(0).astype('int16')
    return df


def fit_multi_scale_bins(df, cols, bin_counts=(10, 50, 100, 200)):
    edges = {}
    for col in cols:
        if col not in df.columns: continue
        edges[col] = {}
        for b in bin_counts:
            try:
                _, e = pd.qcut(df[col], q=b, retbins=True, duplicates='drop')
                edges[col][b] = e
            except ValueError:
                pass
    return edges


def apply_multi_scale_bins(df, edges):
    df = df.copy()
    for col, scales in edges.items():
        if col not in df.columns: continue
        for b, e in scales.items():
            name = f'{col}_msb{b}'
            df[name] = pd.cut(df[col], bins=e, labels=False, include_lowest=True)
            df[name] = df[name].fillna(-1).astype('int16')
    return df


def fit_rank_maps(df, cols):
    maps = {}
    for col in cols:
        if col not in df.columns: continue
        vals = df[col].sort_values().values
        ranks = pd.Series(vals).rank(method='average', pct=True).values
        maps[col] = dict(zip(vals, ranks))
    return maps


def apply_rank_maps(df, maps):
    df = df.copy()
    for col, m in maps.items():
        if col in df.columns:
            r = df[col].map(m)
            med = r.median() if r.notna().any() else 0.5
            df[f'{col}_rank_pct'] = r.fillna(med).astype('float32')
    return df


def add_extended_digit_features(df, cols, k_range=range(-4, 4)):
    """Aggressive digit-feature mining: extracts the digit at multiple decimal scales.
    Range erweitert von (-2, 2) → (-4, 4) basierend auf yunsuxiaozi 0.97981 CV."""
    df = df.copy()
    for col in cols:
        if col not in df.columns: continue
        v = df[col].astype('float64')
        for k in k_range:
            df[f'{col}_d{k}'] = (np.floor(v * (10**k) + 1e-9) % 10).fillna(-1).astype('int8')
    return df


# ============================================================
# Apply pipeline
# ============================================================
_temp_le = LabelEncoder()
train[TARGET] = _temp_le.fit_transform(train[TARGET])
_target_classes = _temp_le.classes_

print('Phase 1: Base features (threshold + non-linear + domain) ...')
train = add_base_features(train)
test  = add_base_features(test)
print(f'  -> train {train.shape}, test {test.shape}')

print('Phase 2: Tree-binning (fit on train target) ...')
tree_edges = fit_tree_bins(train, train[TARGET], BASE_NUM_COLS)
train = apply_tree_bins(train, tree_edges)
test  = apply_tree_bins(test, tree_edges)
print(f'  -> train {train.shape}, test {test.shape}')

print('Phase 3: Multi-scale quantile bins (10/50/100/200) ...')
msb_edges = fit_multi_scale_bins(train, BASE_NUM_COLS, bin_counts=(10, 50, 100, 200))
train = apply_multi_scale_bins(train, msb_edges)
test  = apply_multi_scale_bins(test, msb_edges)
print(f'  -> train {train.shape}, test {test.shape}')

print('Phase 4: Rank-percentile encoding (fit on combined) ...')
full_num = pd.concat([train[BASE_NUM_COLS], test[BASE_NUM_COLS]], ignore_index=True)
rank_maps = fit_rank_maps(full_num, BASE_NUM_COLS)
train = apply_rank_maps(train, rank_maps)
test  = apply_rank_maps(test, rank_maps)
del full_num
print(f'  -> train {train.shape}, test {test.shape}')

print('Phase 5: Extended digit features (range -4..4) ...')
train = add_extended_digit_features(train, BASE_NUM_COLS, k_range=range(-4, 4))
test  = add_extended_digit_features(test,  BASE_NUM_COLS, k_range=range(-4, 4))
print(f'  -> train {train.shape}, test {test.shape}')

train[TARGET] = _temp_le.inverse_transform(train[TARGET].astype(int))

print(f'\n[OK] Feature engineering done. Train: {train.shape} | Test: {test.shape}')


## 4. Encoding

In [ ]:
# ============================================================
# Encoding + OrderedTE (Catboost-style) + Pair Combinations
# Replaces K-fold TE with leakage-free ordered TE from yunsuxiaozi 0.97981.
# ============================================================

# ── 1) Target encoding ──
le_target = LabelEncoder()
train[TARGET] = le_target.fit_transform(train[TARGET])
print(f'Class mapping: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')

# ── 2) Frequency-based label encoding for object cols + bin cols ──
# Categories appearing < 5 times collapsed into a single bucket.
# Inspiriert vom yunsuxiaozi notebook.
print('\nFrequency label-encoding object + bin columns ...')
obj_cols = [c for c in train.columns if train[c].dtype == 'object']
bin_cols = [c for c in train.columns if '_d' in c[-3:] or '_tbin' in c or '_msb' in c]
to_encode = obj_cols + [c for c in bin_cols if train[c].dtype != 'object']

for col in obj_cols:
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    freq = combined.value_counts()
    keep = freq[freq >= 5].index
    mapping = {v: i for i, v in enumerate(keep)}
    default = len(mapping)
    train[col] = train[col].astype(str).map(lambda x: mapping.get(x, default)).astype('int32')
    test[col]  = test[col].astype(str).map(lambda x: mapping.get(x, default)).astype('int32')

# ── 3) Frequency encoding (count features) ──
freq_cols = [c for c in train.columns if '_msb' in c or '_tbin' in c]
print(f'Frequency-encoding {len(freq_cols)} bin columns')
for col in freq_cols:
    combined = pd.concat([train[col], test[col]], axis=0)
    counts = combined.value_counts(normalize=True)
    train[f'{col}_freq'] = train[col].map(counts).astype('float32')
    test[f'{col}_freq']  = test[col].map(counts).astype('float32')

# ── 4) OrderedTE (Catboost-style cumulative target encoding) ──
class OrderedTE:
    """Catboost-style ordered target encoding without K-fold leakage.
    For each category, computes cumulative target probability per class,
    excluding the current row. Multi-class output: 3 columns per feature."""
    def __init__(self, smoothing=20):
        self.smoothing = smoothing
        self.classes_ = None
        self.global_prior_ = None
        self.stats_ = {}

    def fit_transform(self, train_df, cols, target_col, seed=42):
        # Shuffle to break order bias, but record original index to restore order
        shuffled = train_df.sample(frac=1, random_state=seed).copy()
        orig_idx = shuffled.index.values

        self.classes_ = sorted(train_df[target_col].unique())
        self.global_prior_ = (
            train_df[target_col].value_counts(normalize=True).sort_index().values
        )

        out_train = pd.DataFrame(index=shuffled.index)
        for col in cols:
            if col not in shuffled.columns:
                continue
            for k, cls in enumerate(self.classes_):
                y_bin = (shuffled[target_col] == cls).astype('float32').values
                df = pd.DataFrame({col: shuffled[col].values, 'y': y_bin})
                df['cnt']     = 1
                df['cum_cnt'] = df.groupby(col)['cnt'].cumsum() - 1
                df['cum_sum'] = df.groupby(col)['y'].cumsum() - df['y']
                prior = self.global_prior_[k]
                te = (df['cum_sum'] + self.smoothing * prior) / (df['cum_cnt'] + self.smoothing)
                te[df['cum_cnt'] == -1] = prior  # first occurrence
                out_train[f'{col}_te{k}'] = te.values.astype('float32')

            # Save full-data stats for the test transform
            stats = train_df.groupby(col)[target_col].agg(['count']).rename(columns={'count': 'cnt'})
            for k, cls in enumerate(self.classes_):
                m = (train_df[target_col] == cls).groupby(train_df[col]).sum()
                stats[f'sum{k}'] = m
                stats[f'sum{k}'] = stats[f'sum{k}'].fillna(0)
            self.stats_[col] = stats.fillna(0)

        # Restore original train order
        out_train = out_train.loc[train_df.index]
        return pd.concat([train_df, out_train], axis=1)

    def transform(self, test_df, cols):
        out = pd.DataFrame(index=test_df.index)
        for col in cols:
            if col not in test_df.columns or col not in self.stats_:
                continue
            stats = self.stats_[col]
            cnt = test_df[col].map(stats['cnt']).fillna(0).values
            for k in range(len(self.classes_)):
                prior = self.global_prior_[k]
                s = test_df[col].map(stats[f'sum{k}']).fillna(0).values
                te = (s + self.smoothing * prior) / (cnt + self.smoothing)
                te = np.where(cnt == 0, prior, te)
                out[f'{col}_te{k}'] = te.astype('float32')
        return pd.concat([test_df, out], axis=1)


# Apply OrderedTE on Tier-1 features
TE_T1_CANDIDATES = [
    'Crop_Growth_Stage', 'Mulching_Used', 'Crop_Type', 'Season',
    'Soil_Type', 'Irrigation_Type', 'Water_Source', 'Region',
    'Soil_Moisture_msb10', 'Soil_Moisture_tbin', 'is_Soil_Moisture',
    'Wind_Speed_kmh_msb10', 'Wind_Speed_kmh_tbin', 'is_Wind_Speed_kmh',
    'Temperature_C_msb10', 'Temperature_C_tbin', 'is_Temperature_C',
    'Rainfall_mm_msb10', 'Rainfall_mm_tbin',
    'Humidity_msb10',
]
TE_T1 = [c for c in TE_T1_CANDIDATES if c in train.columns]
print(f'\nOrderedTE on {len(TE_T1)} features')

ote = OrderedTE(smoothing=20)
train = ote.fit_transform(train, TE_T1, TARGET)
test  = ote.transform(test, TE_T1)

# ── 5) Pair combinations → OrderedTE → drop raw pair ──
PAIR_SOURCES_CANDIDATES = [
    'Crop_Growth_Stage',
    'Soil_Moisture_msb10', 'Wind_Speed_kmh_tbin', 'Mulching_Used',
    'Temperature_C_msb10', 'is_Soil_Moisture',
]
PAIR_SOURCES = [c for c in PAIR_SOURCES_CANDIDATES if c in train.columns]
print(f'\nPair combinations from {len(PAIR_SOURCES)} sources')

pair_cols = []
for a, b in combinations(PAIR_SOURCES, 2):
    name = f'pair_{a}__{b}'
    k = int(max(train[b].max(), test[b].max())) + 1
    train[name] = (train[a].astype('int64') * k + train[b].astype('int64')).astype('int64')
    test[name]  = (test[a].astype('int64') * k + test[b].astype('int64')).astype('int64')
    pair_cols.append(name)

ote_pair = OrderedTE(smoothing=20)
train = ote_pair.fit_transform(train, pair_cols, TARGET)
test  = ote_pair.transform(test, pair_cols)
train = train.drop(columns=pair_cols)
test  = test.drop(columns=pair_cols)

# ── 6) Build feature matrix ──
FEATURES = [c for c in train.columns if c not in ['id', TARGET]]
X      = train[FEATURES].copy()
y      = train[TARGET].copy()
X_test = test[FEATURES].copy()

still_obj = [c for c in X.columns if X[c].dtype == 'object']
if still_obj:
    X = X.drop(columns=still_obj); X_test = X_test.drop(columns=still_obj)
    FEATURES = [c for c in FEATURES if c not in still_obj]

X = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
nan_cols = [c for c in FEATURES if X[c].isna().any() or X_test[c].isna().any()]
for c in nan_cols:
    med = X[c].median()
    if pd.isna(med): med = 0
    X[c] = X[c].fillna(med); X_test[c] = X_test[c].fillna(med)
if nan_cols:
    print(f'Filled NaN/inf in {len(nan_cols)} columns')

# Drop near-constant
nunique = X.nunique()
const_cols = nunique[nunique <= 1].index.tolist()
if const_cols:
    print(f'Dropping {len(const_cols)} near-constant columns')
    X = X.drop(columns=const_cols); X_test = X_test.drop(columns=const_cols)
    FEATURES = [c for c in FEATURES if c not in const_cols]

print(f'\n[OK] Feature matrix ready')
print(f'   FEATURES: {len(FEATURES)}')
print(f'   X: {X.shape} | y: {y.shape} | X_test: {X_test.shape}')


## 5. LightGBM — 5-Fold Stratified CV

In [ ]:
# LightGBM training as a function so we can re-run after pseudo-labeling
lgb_params_base = {
    'objective':         'multiclass',
    'num_class':         3,
    'metric':            'multi_logloss',
    'learning_rate':     0.03,            # niedriger als vorher (0.05)
    'num_leaves':        63,              # weniger als 127 → weniger overfit
    'max_depth':         7,
    'min_child_samples': 50,              # mehr Regularisierung
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      1,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'n_estimators':      N_ITERS,
    'verbose':           -1,
    'class_weight':      'balanced',
}

def train_lgb(X_in, y_in, X_test_in, sample_weight=None):
    oof   = np.zeros((len(X_in), 3))
    preds = np.zeros((len(X_test_in), 3))
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_in, y_in)):
            X_tr, X_val = X_in.iloc[tr_idx], X_in.iloc[val_idx]
            y_tr, y_val = y_in.iloc[tr_idx], y_in.iloc[val_idx]
            sw_tr = sample_weight[tr_idx] if sample_weight is not None else None

            model = lgb.LGBMClassifier(**lgb_params_base, random_state=seed)
            model.fit(
                X_tr, y_tr, sample_weight=sw_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
            )
            oof[val_idx] += model.predict_proba(X_val) / len(SEEDS)
            preds        += model.predict_proba(X_test_in) / (N_SPLITS * len(SEEDS))
            fold_scores.append(balanced_accuracy_score(
                y_val, np.argmax(model.predict_proba(X_val), axis=1)))
        print(f'  LGB seed {seed} | folds: {[round(s,5) for s in fold_scores]}')
    return oof, preds, model  # last model for feat-importance

print('── LightGBM Round 1 ──')
lgb_oof, lgb_preds, lgb_last_model = train_lgb(X, y, X_test)
lgb_cv = balanced_accuracy_score(y, np.argmax(lgb_oof, axis=1))
print(f'\n✅ LightGBM OOF: {lgb_cv:.5f}')

In [ ]:
# Feature importance from last LGB model
importances = pd.DataFrame({
    'feature': FEATURES,
    'importance': lgb_last_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
top_n = min(30, len(importances))
sns.barplot(data=importances.head(top_n), x='importance', y='feature', palette='viridis')
plt.title(f'LightGBM — Top {top_n} Feature Importances', fontsize=13)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(importances.head(10).to_string(index=False))

## 6. CatBoost — 5-Fold Stratified CV

In [ ]:
cb_params_base = dict(
    iterations=N_ITERS,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=5,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro',
    early_stopping_rounds=100,
    verbose=0,
    auto_class_weights='Balanced',
    task_type='CPU',
)

def train_cb(X_in, y_in, X_test_in):
    oof   = np.zeros((len(X_in), 3))
    preds = np.zeros((len(X_test_in), 3))
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_in, y_in)):
            X_tr, X_val = X_in.iloc[tr_idx], X_in.iloc[val_idx]
            y_tr, y_val = y_in.iloc[tr_idx], y_in.iloc[val_idx]

            model = CatBoostClassifier(**cb_params_base, random_seed=seed)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            oof[val_idx] += model.predict_proba(X_val) / len(SEEDS)
            preds        += model.predict_proba(X_test_in) / (N_SPLITS * len(SEEDS))
            fold_scores.append(balanced_accuracy_score(
                y_val, np.argmax(model.predict_proba(X_val), axis=1)))
        print(f'  CB seed {seed} | folds: {[round(s,5) for s in fold_scores]}')
    return oof, preds

print('── CatBoost Round 1 ──')
cb_oof, cb_preds = train_cb(X, y, X_test)
cb_cv = balanced_accuracy_score(y, np.argmax(cb_oof, axis=1))
print(f'\n✅ CatBoost OOF: {cb_cv:.5f}')

## 6b. XGBoost — 5-Fold Stratified CV

In [ ]:
# XGBoost — params inspired by yunsuxiaozi 0.97981 CV
# Key insights from that notebook:
# - max_depth=4 (shallow!) but max_leaves=30 (still expressive)
# - high alpha=5 + reg_lambda=5 (heavy regularization)
# - max_bin=10000 (very fine binning)
# - tree_method='hist'
# - sample_weight from class balance
xgb_params_base = dict(
    objective='multi:softprob',
    num_class=3,
    learning_rate=0.05,
    max_depth=4,
    max_leaves=30,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=5,
    reg_lambda=5,
    n_estimators=512,
    eval_metric='mlogloss',
    tree_method='hist',
    max_bin=10000,
    verbosity=0,
)


def train_xgb(X_in, y_in, X_test_in):
    oof   = np.zeros((len(X_in), 3))
    preds = np.zeros((len(X_test_in), 3))
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_in, y_in)):
            X_tr, X_val = X_in.iloc[tr_idx], X_in.iloc[val_idx]
            y_tr, y_val = y_in.iloc[tr_idx], y_in.iloc[val_idx]
            sw_tr = compute_sample_weight(class_weight='balanced', y=y_tr)

            model = xgb.XGBClassifier(**xgb_params_base, random_state=seed)
            model.fit(X_tr, y_tr, sample_weight=sw_tr, verbose=False)

            oof[val_idx] += model.predict_proba(X_val) / len(SEEDS)
            preds        += model.predict_proba(X_test_in) / (N_SPLITS * len(SEEDS))
            fold_scores.append(balanced_accuracy_score(
                y_val, np.argmax(model.predict_proba(X_val), axis=1)))
        print(f'  XGB seed {seed} | folds: {[round(s,5) for s in fold_scores]}')
    return oof, preds


print('-- XGBoost Round 1 --')
xgb_oof, xgb_preds = train_xgb(X, y, X_test)
xgb_cv = balanced_accuracy_score(y, np.argmax(xgb_oof, axis=1))
print(f'\n[OK] XGBoost OOF: {xgb_cv:.5f}')

# Safety net: drop XGB from blend if score is more than 0.01 below LGB
XGB_FLOOR = lgb_cv - 0.01
USE_XGB = xgb_cv >= XGB_FLOOR
if not USE_XGB:
    print(f'[WARN] XGB ({xgb_cv:.5f}) below floor ({XGB_FLOOR:.5f}) -- excluded from ensemble')
else:
    print(f'[OK] XGB above floor -- included in ensemble')


## 7. Weighted Ensemble

In [ ]:
print(f'LightGBM OOF: {lgb_cv:.5f}')
print(f'CatBoost OOF: {cb_cv:.5f}')
print(f'XGBoost  OOF: {xgb_cv:.5f}  (use={USE_XGB})')

# ── 1) Blend search (2-way if XGB is off, 3-way otherwise) ──
if USE_XGB:
    def neg_balacc_n(w_arr, oofs, y_true):
        w = np.clip(w_arr, 0.0, None)
        if w.sum() < 1e-9: return 0.0
        w = w / w.sum()
        blended = sum(wi * oi for wi, oi in zip(w, oofs))
        return -balanced_accuracy_score(y_true, np.argmax(blended, axis=1))

    best = minimize(neg_balacc_n, x0=[1/3, 1/3, 1/3],
                    args=([lgb_oof, cb_oof, xgb_oof], y),
                    method='Nelder-Mead', options={'xatol': 1e-3, 'fatol': 1e-5, 'maxiter': 800})
    w = np.clip(best.x, 0, None); w = w / w.sum()
    print(f'\nBlend weights → LGB={w[0]:.3f} | CB={w[1]:.3f} | XGB={w[2]:.3f}')
    ens_oof   = w[0]*lgb_oof   + w[1]*cb_oof   + w[2]*xgb_oof
    ens_preds = w[0]*lgb_preds + w[1]*cb_preds + w[2]*xgb_preds
else:
    # 2-way blend LGB+CB only
    def neg_balacc_2(w_arr, oa, ob, y_true):
        w = np.clip(w_arr[0], 0.0, 1.0)
        return -balanced_accuracy_score(y_true, np.argmax(w*oa + (1-w)*ob, axis=1))
    best = minimize(neg_balacc_2, x0=[0.5], args=(lgb_oof, cb_oof, y),
                    method='Nelder-Mead', options={'xatol': 1e-3, 'fatol': 1e-5})
    w_lgb = float(np.clip(best.x[0], 0.0, 1.0))
    print(f'\n2-way Blend → LGB={w_lgb:.3f} | CB={1-w_lgb:.3f}')
    ens_oof   = w_lgb*lgb_oof   + (1-w_lgb)*cb_oof
    ens_preds = w_lgb*lgb_preds + (1-w_lgb)*cb_preds

blend_score = balanced_accuracy_score(y, np.argmax(ens_oof, axis=1))
print(f'🏆 Blend OOF: {blend_score:.5f}')

# ── 2) Stacking meta-model (LogReg) — nested K-fold ──
if USE_STACKING:
    print('\n── Stacking meta-model (LogReg) ──')
    if USE_XGB:
        meta_X      = np.hstack([lgb_oof, cb_oof, xgb_oof]).astype(np.float32)
        meta_X_test = np.hstack([lgb_preds, cb_preds, xgb_preds]).astype(np.float32)
    else:
        meta_X      = np.hstack([lgb_oof, cb_oof]).astype(np.float32)
        meta_X_test = np.hstack([lgb_preds, cb_preds]).astype(np.float32)

    stack_oof = np.zeros_like(lgb_oof)
    skf_meta  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for tr_idx, val_idx in skf_meta.split(meta_X, y):
        meta = LogisticRegression(
            max_iter=2000, C=1.0, class_weight='balanced',
            multi_class='multinomial', solver='lbfgs'
        )
        meta.fit(meta_X[tr_idx], y.iloc[tr_idx])
        stack_oof[val_idx] = meta.predict_proba(meta_X[val_idx])

    stack_score = balanced_accuracy_score(y, np.argmax(stack_oof, axis=1))
    print(f'   Meta OOF (nested CV): {stack_score:.5f}')

    meta_full = LogisticRegression(
        max_iter=2000, C=1.0, class_weight='balanced',
        multi_class='multinomial', solver='lbfgs'
    )
    meta_full.fit(meta_X, y)
    stack_preds = meta_full.predict_proba(meta_X_test)

    if stack_score > blend_score:
        print(f'   → Using stacking ({stack_score:.5f} > {blend_score:.5f})')
        ens_oof, ens_preds = stack_oof, stack_preds
    else:
        print(f'   → Keeping blend ({blend_score:.5f} ≥ {stack_score:.5f})')

# ── 3) Optional Pseudo-Labeling (unchanged, off by default) ──
if USE_PSEUDO_LABELING:
    print('\n── Pseudo-Labeling ──')
    test_max_proba = ens_preds.max(axis=1)
    pseudo_labels  = np.argmax(ens_preds, axis=1)
    pseudo_mask    = test_max_proba >= PSEUDO_CONF_THRESHOLD
    n_pseudo = int(pseudo_mask.sum())
    print(f'  Selected {n_pseudo:,} / {len(pseudo_mask):,} test samples')
    if n_pseudo > 5000:
        X_pseudo = X_test.iloc[pseudo_mask].copy()
        y_pseudo = pd.Series(pseudo_labels[pseudo_mask])
        X_aug = pd.concat([X, X_pseudo], ignore_index=True)
        y_aug = pd.concat([y, y_pseudo], ignore_index=True)
        lgb_oof2, lgb_preds2, _ = train_lgb(X_aug, y_aug, X_test)
        cb_oof2, cb_preds2      = train_cb(X_aug, y_aug, X_test)
        if USE_XGB:
            xgb_oof2, xgb_preds2 = train_xgb(X_aug, y_aug, X_test)
            lgb_o, cb_o, xgb_o = lgb_oof2[:len(y)], cb_oof2[:len(y)], xgb_oof2[:len(y)]
            best2 = minimize(neg_balacc_n, x0=w, args=([lgb_o, cb_o, xgb_o], y),
                             method='Nelder-Mead', options={'xatol':1e-3,'fatol':1e-5,'maxiter':500})
            w2 = np.clip(best2.x, 0, None); w2 = w2 / w2.sum()
            ens_oof_r2 = w2[0]*lgb_o + w2[1]*cb_o + w2[2]*xgb_o
            ens_preds_r2 = w2[0]*lgb_preds2 + w2[1]*cb_preds2 + w2[2]*xgb_preds2
        else:
            lgb_o, cb_o = lgb_oof2[:len(y)], cb_oof2[:len(y)]
            best2 = minimize(neg_balacc_2, x0=[w_lgb], args=(lgb_o, cb_o, y),
                             method='Nelder-Mead', options={'xatol':1e-3,'fatol':1e-5})
            w2_lgb = float(np.clip(best2.x[0], 0.0, 1.0))
            ens_oof_r2 = w2_lgb*lgb_o + (1-w2_lgb)*cb_o
            ens_preds_r2 = w2_lgb*lgb_preds2 + (1-w2_lgb)*cb_preds2
        r2 = balanced_accuracy_score(y, np.argmax(ens_oof_r2, axis=1))
        cur = balanced_accuracy_score(y, np.argmax(ens_oof, axis=1))
        print(f'  Round 2 OOF: {r2:.5f}  (current: {cur:.5f})')
        if r2 > cur:
            ens_oof, ens_preds = ens_oof_r2, ens_preds_r2
            print('  → Using Round 2')

# ── 4) Optional per-class scaling ──
def neg_balacc_scale(scale, oof, y_true):
    s = np.clip(scale, 1e-3, None)
    return -balanced_accuracy_score(y_true, np.argmax(oof * s, axis=1))

res = minimize(neg_balacc_scale, x0=np.ones(3), args=(ens_oof, y),
               method='Nelder-Mead', options={'xatol': 1e-3, 'fatol': 1e-5, 'maxiter': 500})
class_scale = np.clip(res.x, 1e-3, None)
class_scale = class_scale / class_scale.sum() * 3
print(f'\nClass scaling: {dict(zip(le_target.classes_, class_scale.round(3)))}')

ens_score_final = balanced_accuracy_score(y, np.argmax(ens_oof, axis=1))
ens_score_tuned = balanced_accuracy_score(y, np.argmax(ens_oof * class_scale, axis=1))

print('\n── Final Comparison ──')
print(f'LightGBM:           {lgb_cv:.5f}')
print(f'CatBoost:           {cb_cv:.5f}')
print(f'XGBoost:            {xgb_cv:.5f}  (use={USE_XGB})')
print(f'Blend:              {blend_score:.5f}')
if USE_STACKING:
    print(f'Stacking meta:      {stack_score:.5f}')
print(f'Final ensemble:     {ens_score_final:.5f}  ← submit (USE_THRESHOLD_TUNING=False)')
print(f'Ensemble tuned:     {ens_score_tuned:.5f}  ← submit (USE_THRESHOLD_TUNING=True)')

In [ ]:
# OOF Confidence Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_names = le_target.classes_
colors_cls = ['#2ecc71', '#f39c12', '#e74c3c']

for i, (cls, color) in enumerate(zip(class_names, colors_cls)):
    axes[i].hist(ens_oof[:, i], bins=50, color=color, alpha=0.8, edgecolor='white')
    axes[i].set_title(f'Predicted Prob: {cls}', fontsize=11)
    axes[i].set_xlabel('Probability')
    axes[i].set_ylabel('Count')

plt.suptitle('Ensemble OOF Probability Distributions', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Submission

In [ ]:
# ── Threshold-Tuning Toggle ──
# A/B-Test: True = mit class_scale (kann auf synthetischen Daten overfitten)
#           False = raw argmax (robuster auf LB)
USE_THRESHOLD_TUNING = False

if USE_THRESHOLD_TUNING:
    final_class_ids = np.argmax(ens_preds * class_scale, axis=1)
    print(f'Using class_scale: {dict(zip(le_target.classes_, class_scale.round(3)))}')
else:
    final_class_ids = np.argmax(ens_preds, axis=1)
    print('Using raw argmax (no threshold tuning)')

final_labels = le_target.inverse_transform(final_class_ids)

sub[TARGET] = final_labels
sub.to_csv('submission.csv', index=False)

print('submission.csv saved ✅')
print(f'\nPrediction distribution:')
print(sub[TARGET].value_counts())
print(sub[TARGET].value_counts(normalize=True).round(3))
sub.head(10)

## 8b. Public OOF Stacking (Optional)

Loads OOFs from public notebooks if available and overwrites submission.csv.

In [ ]:
# ============================================================
# Public OOF Stacking with TunedBlender
# Inspired by ravi20076 PlaygroundS6E4|TunedBlend|V1 (LB 0.98018).
#
# This cell tries to load OOF predictions from public Kaggle notebooks.
# If found, it averages all model probas + per-fold per-class scaling
# to overwrite submission.csv with a much stronger blend.
#
# REQUIRED INPUTS on Kaggle (Add Input -> Notebook/Dataset):
#   - ravi20076/playgrounds6e4publicmodelsv1            (dataset)
#   - ravi20076/playgrounds6e4-public-baseline-v3       (notebook)
#   - mahoganybuttstrings/pg-s6e4-realmlp-cv-0-97802-lb-0-97685 (notebook)
#   - utaazu/0-979-cv-single-cat-pairwise-te-bias-tuning (notebook)
#   - yunsuxiaozi/pss6e4-lgb-baselinecv-0-97943         (notebook)
#   - yunsuxiaozi/pss6e4-lgb-advanced-cv-0-97997        (notebook)
#   - yunsuxiaozi/pss6e4-xgb-cv-0-979805                (notebook)
#   - wguesdon/ps6e4-irrigation-14-model-predictions    (dataset)
#
# If no inputs found, the existing submission from Cell 8 is kept.
# ============================================================
import os, glob
from itertools import permutations as _perms

def find_best_perm(oof, y_true):
    """Try all 6 column permutations, return the one with best balanced acc."""
    best_perm, best_score = (0, 1, 2), -1
    for perm in _perms(range(3)):
        s = balanced_accuracy_score(y_true, np.argmax(oof[:, list(perm)], axis=1))
        if s > best_score:
            best_score = s
            best_perm = perm
    return list(best_perm), best_score


def load_public_oofs(y_arr, n_test):
    """Scan /kaggle/input for known public OOF files. Return list of (name, oof, test_pred, score)."""
    found = []
    n_train = len(y_arr)
    base = '/kaggle/input'
    if not os.path.exists(base):
        return found

    # (name, oof_glob, test_glob, loader_kind)
    patterns = [
        ('GNN_v4',    '**/OOF_Preds_GNNV4_1.parquet', '**/Mdl_Preds_GNNV4_1.parquet', 'parquet'),
        ('FTT_v4',    '**/OOF_Preds_FTTV4_1.parquet', '**/Mdl_Preds_FTTV4_1.parquet', 'parquet'),
        ('Baseline',  '**/playgrounds6e4-public-baseline-v3/OOF_Preds.npy',
                      '**/playgrounds6e4-public-baseline-v3/Mdl_Preds.npy', 'npy'),
        ('RealMLP',   '**/realmlp_oof.csv', '**/realmlp_pred.csv', 'realmlp_csv'),
        ('Cat_pair',  '**/oof_probs_cat.npy', '**/test_probs_cat.npy', 'npy'),
        ('LGB_base',  '**/pss6e4-lgb-baselinecv-0-97943/oof_preds.npy',
                      '**/pss6e4-lgb-baselinecv-0-97943/test_preds.npy', 'npy'),
        ('LGB_adv',   '**/pss6e4-lgb-advanced-cv-0-97997/oof_preds.npy',
                      '**/pss6e4-lgb-advanced-cv-0-97997/test_preds.npy', 'npy'),
        ('XGB_OTE',   '**/pss6e4-xgb-cv-0-979805/oof_preds.npy',
                      '**/pss6e4-xgb-cv-0-979805/test_preds.npy', 'npy'),
    ]

    for name, oof_pat, test_pat, kind in patterns:
        oof_paths  = glob.glob(os.path.join(base, oof_pat),  recursive=True)
        test_paths = glob.glob(os.path.join(base, test_pat), recursive=True)
        if not oof_paths or not test_paths:
            continue
        try:
            if kind == 'parquet':
                oof = pd.read_parquet(oof_paths[0]).values
                tp  = pd.read_parquet(test_paths[0]).values
            elif kind == 'npy':
                oof = np.load(oof_paths[0])
                tp  = np.load(test_paths[0])
            elif kind == 'realmlp_csv':
                oof = pd.read_csv(oof_paths[0]).values.reshape(n_train, 3)
                tp  = pd.read_csv(test_paths[0]).values.reshape(n_test, 3)
            else:
                continue

            if oof.shape != (n_train, 3) or tp.shape != (n_test, 3):
                print(f'  [skip] {name}: bad shape oof={oof.shape}, test={tp.shape}')
                continue

            perm, score = find_best_perm(oof, y_arr)
            oof = oof[:, perm]
            tp  = tp[:,  perm]
            print(f'  [ok] {name}: shape={oof.shape}, perm={perm}, score={score:.5f}')
            found.append((name, oof, tp, score))
        except Exception as e:
            print(f'  [err] {name}: {e}')

    # 14-model dataset (single dir, many files)
    for oof_path in glob.glob(os.path.join(base, '**/ps6e4-irrigation-14-model-predictions/oof_*.npy'), recursive=True):
        try:
            test_path = oof_path.replace('oof_', 'pred_')
            if not os.path.exists(test_path):
                continue
            oof = np.load(oof_path)
            tp  = np.load(test_path)
            if oof.shape != (n_train, 3) or tp.shape != (n_test, 3):
                continue
            perm, score = find_best_perm(oof, y_arr)
            oof = oof[:, perm]
            tp  = tp[:,  perm]
            name = os.path.basename(oof_path).replace('oof_', '').replace('.npy', '')
            print(f'  [ok] wgu_{name}: perm={perm}, score={score:.5f}')
            found.append((f'wgu_{name}', oof, tp, score))
        except Exception as e:
            print(f'  [err] wguesdon: {e}')

    return found


print('-- Searching for public OOFs --')
y_np = y.values if hasattr(y, 'values') else np.asarray(y)
public_oofs = load_public_oofs(y_np, len(X_test))
print(f'\nFound {len(public_oofs)} public OOF files')

if len(public_oofs) >= 1:
    # Include our own models too
    own_models = [('our_LGB', lgb_oof, lgb_preds, lgb_cv),
                  ('our_CB',  cb_oof,  cb_preds,  cb_cv)]
    if 'USE_XGB' in dir() and USE_XGB:
        own_models.append(('our_XGB', xgb_oof, xgb_preds, xgb_cv))
    all_models = own_models + public_oofs

    # Filter: keep only models within 0.01 of best (drops weak ones)
    best_score = max(s for _, _, _, s in all_models)
    cutoff = best_score - 0.01
    kept = [(n, o, t, s) for n, o, t, s in all_models if s >= cutoff]
    print(f'\nKept {len(kept)}/{len(all_models)} models above cutoff {cutoff:.5f}:')
    for n, _, _, s in sorted(kept, key=lambda x: -x[3]):
        print(f'  {n}: {s:.5f}')

    # Stack
    Xtr_stack = np.hstack([o for _, o, _, _ in kept]).astype('float32')
    Xte_stack = np.hstack([t for _, _, t, _ in kept]).astype('float32')
    n_models  = len(kept)

    # TunedBlender: per-fold optimize per-class scaling
    skf_blend = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_blend = np.zeros((len(y_np), 3), dtype='float32')
    test_per_fold = []
    fold_scales = []
    for tr_idx, val_idx in skf_blend.split(Xtr_stack, y_np):
        avg_tr = Xtr_stack[tr_idx].reshape(-1, n_models, 3).mean(axis=1)
        avg_val = Xtr_stack[val_idx].reshape(-1, n_models, 3).mean(axis=1)
        avg_te  = Xte_stack.reshape(-1, n_models, 3).mean(axis=1)
        y_tr    = y_np[tr_idx]

        def _neg(scale):
            s = np.clip(scale, 1e-3, None)
            return -balanced_accuracy_score(y_tr, np.argmax(avg_tr * s, axis=1))

        res = minimize(_neg, x0=np.ones(3), method='Nelder-Mead',
                       options={'xatol': 1e-4, 'fatol': 1e-6, 'maxiter': 2000})
        scale = np.clip(res.x, 1e-3, None)
        fold_scales.append(scale)
        oof_blend[val_idx] = avg_val * scale
        test_per_fold.append(avg_te * scale)

    test_blend = np.mean(test_per_fold, axis=0)
    print('\nPer-fold tuned scales:')
    for i, sc in enumerate(fold_scales):
        print(f'  Fold {i+1}: {sc.round(5)}')
    print(f'  Mean   : {np.mean(fold_scales, axis=0).round(5)}')

    blend_score = balanced_accuracy_score(y_np, np.argmax(oof_blend, axis=1))
    print(f'\n[BLEND] Public-OOF tuned blend OOF: {blend_score:.5f}')

    # Overwrite submission
    final_class_ids = np.argmax(test_blend, axis=1)
    final_labels = le_target.inverse_transform(final_class_ids)
    sub[TARGET] = final_labels
    sub.to_csv('submission.csv', index=False)
    print(f'\n[OK] submission.csv overwritten with public-OOF tuned blend')
    print(sub[TARGET].value_counts().to_dict())
else:
    print('\n[INFO] No public OOFs found -- keeping standard ensemble submission from Cell 8')


## 9. (Optional) Optuna Hyperparameter Tuning

In [ ]:
# Nur ausführen wenn du Zeit hast — dauert 30–60 min
RUN_OPTUNA = False

if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    skf_tune = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    def objective(trial):
        params = {
            'objective':         'multiclass',
            'num_class':         3,
            'metric':            'multi_logloss',
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
            'max_depth':         trial.suggest_int('max_depth', 4, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq':      1,
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            'n_estimators':      2000,
            'random_state':      SEED,
            'verbose':           -1,
            'class_weight':      'balanced',
        }
        oof = np.zeros((len(train), 3))
        for tr_idx, val_idx in skf_tune.split(X, y):
            m = lgb.LGBMClassifier(**params)
            m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
                  eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
            oof[val_idx] = m.predict_proba(X.iloc[val_idx])
        return balanced_accuracy_score(y, np.argmax(oof, axis=1))

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    print(f'\nBest balanced accuracy: {study.best_value:.5f}')
    print(f'Best params: {study.best_params}')